# Wedding Planner 
Un agent principal (main coordinator) qui utilise des sous agent(subagents) pour planifier le mariage idéal 
- Un **agent de voyage** qui recherche des vols vers et depuis votre destination idéale.
- Un **agent de recherche de lieux** qui parcourt le Web pour trouver un lieu de mariage.
- Un **agent DJ** qui parcourt une base de données musicale afin de créer une playlist correspondant au genre musical souhaité.
- Un **agent coordinateur principal** qui rassemble et coordonne l’ensemble pour concrétiser la vision globale.

In [69]:
from dotenv import load_dotenv
load_dotenv()

True

In [70]:
from dataclasses import dataclass
from http import client
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.tools import tool
model = ChatOpenAI(model="gpt-4.1-mini")
from langchain.messages import HumanMessage
import os



In [71]:
#Agent de voyages pour la recherche de vols et destinations idéal
@dataclass
class TravelContext:
    destination: str = "Hawaii"
    departure: str = "Paris"
    departure_date: str = "2026-12-12"
    budget: float = 2000.0
    currency: str = "EUR"
    persons: int = 2

travel_client = MultiServerMCPClient(
    {
        "kiwi": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    }
)


travel_tools = await travel_client.get_tools()

for t in travel_tools:
    print(t.name)


search-flight
feedback-to-devs


In [72]:
#création de l'agent de voyage 
Tagent = create_agent(
    model=model,
    tools=travel_tools,
    context_schema=TravelContext,
    system_prompt="""
    You are a flight search agent.

    Use the available flight search tool to find flights matching
    the user's requirements.

    Return:
    - departure and arrival airports
    - date
    - price
    - duration
    - stops
    - the exact booking URL returned by the tool

    Never invent a booking URL.
    """
)

In [73]:
#Agent de recherche de lieux qui parcours le wb pour trouver un lieux de marige 
from typing import Dict, Any 
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool("web_search")
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for wedding venue information."""
    return tavily_client.search(query)





In [74]:
Wagent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="""
    You are a wedding venue search agent.

    Search the web for a real wedding venue matching the user's
    location, guest capacity, budget and requested features.

    Return:
    - venue name
    - location
    - guest capacity
    - price if available
    - parking information
    - reception facilities
    - exact website or source URL returned by the search tool

    If price or availability cannot be verified, say so.
    Never invent missing information or URLs.
    """
)

In [75]:
# Agent de musique (DJ)
# Agent de musique (Spotify)

@dataclass
class MusicContext:
    genre: str = "Afrobeat"
    mood: str = "Wedding"
    number_of_tracks: int = 75
    language: str = "Any"



HASDATA_API_KEY = os.getenv("HASDATA_API_KEY")

youtube_client = MultiServerMCPClient(
    {
        "youtube": {
            "transport": "streamable_http",
            "url": "https://mcp.hasdata.com/mcp?apis=youtube",
            "headers": {
                "x-api-key": HASDATA_API_KEY
            }
        }
    }
)

wanted_tools = {
    "hasdata_youtube_search_getYoutubeSearchResults",
    "hasdata_youtube_video_getYoutubeVideo",
}

music_tools = [
    t for t in youtube_tools
    if t.name in wanted_tools
]

for t in music_tools:
    print(t.name)

hasdata_youtube_search_getYoutubeSearchResults
hasdata_youtube_video_getYoutubeVideo


In [76]:
MusicAgent = create_agent(
    model=model,
    tools=music_tools,
    context_schema=MusicContext,
    system_prompt="""
You are a wedding music agent.

Your role is to find music on YouTube for a wedding based on the
user's musical preferences.

Always use the available YouTube search tools.

The wedding music should respect the provided context:
- preferred genre
- wedding mood
- preferred language
- requested number of tracks

Search YouTube for appropriate songs and build a wedding music selection.

For every song, return:
- artist
- song title
- YouTube URL when available from the tool

Prioritize real songs returned by the YouTube search tool.
Do not invent YouTube URLs.

The final selection should contain the requested number of tracks
when enough relevant results are available.

Return the result as:

WEDDING PLAYLIST

1. Artist - Song
   YouTube: URL

2. Artist - Song
   YouTube: URL

...

Your result will be sent to the main wedding coordinator.
"""
)

In [77]:
MusicAgent = create_agent(
    model=model,
    tools=music_tools,
    context_schema=MusicContext,
    system_prompt="""
You are a wedding music research agent.

Your task is to build a complete list of individual songs for a wedding
using the available YouTube search tools.

IMPORTANT:
You MUST use the YouTube search tool.
Do not rely only on your internal knowledge.

SEARCH STRATEGY:

Perform multiple YouTube searches using different queries when necessary.

For an Afrobeat wedding, for example, search for:
- Afrobeat wedding songs
- Afrobeats love songs
- Nigerian wedding songs
- Ghanaian wedding songs
- Afrobeat romantic songs
- Afrobeats dance songs
- Afrobeat party songs
- African wedding songs

Continue searching until you have enough unique individual songs
to satisfy the requested number of tracks.

SELECTION RULES:

- Return individual songs, not DJ mixes or compilations.
- Avoid duplicate songs.
- Prefer official music videos or official artist uploads.
- Songs must be appropriate for a wedding.
- Include romantic songs as well as energetic/dance songs.
- Respect the requested genre and mood.

For each song return:
- artist
- song title
- exact YouTube URL returned by the tool

Never invent a YouTube URL.

If the user requests 75 songs, your goal is to return exactly 75
unique individual songs.

Do NOT stop after the first search.
Do NOT tell the user to continue searching themselves.
Use additional searches when the first search does not provide
enough results.

Only return fewer songs if the available YouTube search results
are genuinely insufficient. If that happens, clearly state how many
verified songs were found.

Format:

WEDDING PLAYLIST

1. Artist - Song
   YouTube: URL

2. Artist - Song
   YouTube: URL

...

75. Artist - Song
    YouTube: URL
"""
)

In [78]:
# Appel des différents sous-agents
@tool
async def search_flights(request: str) -> str:
    """Search for flights for the wedding."""
    
    response = await Tagent.ainvoke(
        {"messages": [HumanMessage(content=request)]}
    )
    
    return response["messages"][-1].content


@tool
async def search_wedding_venue(request: str) -> str:
    """
    Call the wedding venue agent to search for a wedding venue.
    Use this tool when a venue needs to be found for the wedding.
    """
    response = await Wagent.ainvoke(
        {
            "messages": [
                HumanMessage(content=request)
            ]
        }
    )

    return response["messages"][-1].content


@tool
async def search_wedding_music(request: str) -> str:
    """Search YouTube for music suitable for the wedding."""
    
    response = await MusicAgent.ainvoke(
        {
            "messages": [
                HumanMessage(content=request)
            ]
        }
    )
    
    return response["messages"][-1].content


In [79]:
coordinator = create_agent(
    model=model,
    tools=[
        search_flights,
        search_wedding_venue,
        search_wedding_music,
    ],
    system_prompt="""
You are the main coordinator for a wedding planning system.

Your role is to coordinate specialized agents in order to plan
the user's wedding.

You have three specialized agents available through tools:

1. search_flights:
   Use this agent to search for flights.

2. search_wedding_venue:
   Use this agent to search for a wedding venue.

3. search_wedding_music:
   Use this agent to find music on YouTube for the wedding.

Read the user's wedding requirements carefully.

Pass all relevant information to each specialized agent.

For the music agent, pass:
- preferred music genre
- wedding mood
- preferred language
- requested number of tracks

Call the appropriate specialized agents when their services are needed.

After receiving the results from all specialized agents,
combine them into one clear wedding plan.

Do not invent flights, venues, songs, prices, availability or URLs.
Only use information returned by the specialized agents.
"""
)

In [80]:
question = """
I want to plan a wedding in Hawaii on 2026-12-12.

Here are my requirements:

- 150 wedding guests
- Flights from Paris to Hawaii for 2 people
- Maximum flight budget: 2000 EUR
- Maximum wedding venue budget: 8000 EUR
- The venue should preferably be seaside
- The venue should have parking
- The venue should have a reception room
- Preferred music genre: Afrobeat
- Music mood: Wedding
- Number of songs: 75
- Music language: Any

Please find:
1. Flights
2. A wedding venue
3. A selection of 75 wedding songs using YouTube

For the music selection, include the artist, song title and YouTube URL
when available from the search results.

Then create a complete wedding plan using the results.
"""
response = await coordinator.ainvoke(
    {
        "messages": [
            HumanMessage(content=question)
        ]
    }
)

In [81]:
from pprint import pprint

pprint(response["messages"][-1].content)

('Here is the complete wedding plan based on your requirements for a wedding '
 'in Hawaii on 2026-12-12:\n'
 '\n'
 '1. Flights from Paris to Hawaii for 2 people within 2000 EUR budget:\n'
 '- Cheapest option: Paris Orly (ORY) → Newark (EWR) → Honolulu (HNL), economy '
 'class for 1462 EUR total, duration approx. 31h 15m. Booking available here: '
 'https://kiwi.com/u/hgsgctr\n'
 '- Other options with slightly different routes and prices are also '
 'available, up to around 1651 EUR.\n'
 '\n'
 '2. Wedding venue in Hawaii for 150 guests, budget max 8000 EUR, seaside '
 'preferred, parking and reception room:\n'
 '- Venues fitting all criteria and budget exactly are difficult to confirm '
 'via current data.\n'
 '- Recommended venues for inquiry are:\n'
 '   - Lanikuhonua: oceanfront, parking available, covered pavilions, fits '
 'approx. 150 guests.\n'
 '   - Moana Surfrider in Waikiki: oceanfront hotel with wedding venues and '
 'amenities.\n'
 '- Average wedding venue costs for 150 gu

### les problèmes rencontrés
Le principal problème venait d’abord de **l’environnement Python** : le notebook utilisait un `.venv` dans lequel `langchain_mcp_adapters` n’était pas installé. Après avoir recréé et correctement configuré l’environnement, la connexion au serveur MCP HasData renvoyait une erreur **401 Unauthorized**, corrigée avec le bon endpoint et l’authentification par clé API. Dans la définition des outils appelant les subagents, j’avais également utilisé par erreur le type **`float`** pour les paramètres et les valeurs de retour, alors que les agents échangent des requêtes et des réponses textuelles : ces types ont donc été corrigés en **`str`**. Enfin, comme les outils MCP sont **asynchrones(async def)**, les appels `.invoke()` provoquaient l’erreur `StructuredTool does not support sync invocation` ; ils ont été remplacés par `await ... .ainvoke()`. Après ces corrections, les trois subagents (vol, lieu de mariage et DJ) peuvent être orchestrés par le coordinateur. Le principal problème restant concerne certains **liens retournés qui mènent à des pages 404**, et l'ajout de règles dans les system prompts n'a pas suffi à résoudre complètement ce comportement.